# 03. reverse B-S回帰（Penn effect）

このノートブックでは、(1) プール回帰の標準誤差を古典的・HC3・国クラスタ・二方向クラスタで計算し、
(2) 年固定効果付きプール回帰、(3) 年別の横断面回帰を行い、時系列の解釈を年効果なしプール回帰の
2時点比較だけに依拠しない形にする。(4) 除外国リストの有無に対する感度分析も行う。

**対米相対所得（`reli`）に対米相対物価水準（`relp`）を回帰する**（log-log）。
傾きが正であれば「所得が高い国ほど物価水準も高い」という通常のBalassa-Samuelson
効果の予測と整合する（**理論が予測するのは正の関係であり、傾きが1になることでは
ない**）。残差は「所得水準から予想される物価水準からのズレ」を表す。

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_TABLES_DIR = BASE_DIR / "output" / "tables"
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE_DIR))

from src.regression_models import (
    classical_se, cluster_se, fit_penn_regression, fit_penn_regression_by_year,
    fit_penn_regression_with_year_fe, hc3_se, ols, trim_ols, twoway_cluster_se,
)

FOCUS_COUNTRIES = {"JPN": "日本", "KOR": "韓国", "TWN": "台湾"}

pool = pd.read_csv(PROCESSED_DIR / "ppp_panel_pooled_2005_2023.csv")
cross_2023_world = pd.read_csv(PROCESSED_DIR / "ppp_cross_section_2023_world.csv")
cross_2023_hi = pd.read_csv(PROCESSED_DIR / "ppp_cross_section_2023_highincome.csv")

print(f"プールパネル: {len(pool):,}国年、{pool['countrycode'].nunique()}か国")
print(f"2023年世界全体: {len(cross_2023_world)}か国")
print(f"2023年高所得クラブ: {len(cross_2023_hi)}か国")

プールパネル: 621国年、41か国
2023年世界全体: 185か国
2023年高所得クラブ: 40か国


## 1. 世界全体（2023年、185か国）での回帰

In [2]:
world_fit = fit_penn_regression(cross_2023_world)
print(f"世界全体（2023年）: slope={world_fit.attrs['slope']:.3f}  R2={world_fit.attrs['r_squared']:.3f}  n={len(world_fit)}")
for cc, name in FOCUS_COUNTRIES.items():
    row = world_fit[world_fit["countrycode"] == cc].iloc[0]
    print(f"  {name}: resid={row['resid']:+.3f} ({row['resid_pct']:+.1f}%)")

世界全体（2023年）: slope=0.135  R2=0.082  n=185
  日本: resid=+0.296 (+34.4%)
  韓国: resid=+0.177 (+19.4%)
  台湾: resid=-0.246 (-21.8%)


## 2. 高所得クラブ（2023年単年）での回帰

In [3]:
hi_fit = fit_penn_regression(cross_2023_hi)
hi_se_hc3 = hc3_se(hi_fit["log_x"].to_numpy(), hi_fit["log_y"].to_numpy())
print(f"高所得クラブ（2023年）: slope={hi_fit.attrs['slope']:.3f}（HC3 SE={hi_se_hc3:.3f}）  R2={hi_fit.attrs['r_squared']:.3f}  n={len(hi_fit)}")

for cc, name in FOCUS_COUNTRIES.items():
    row = hi_fit[hi_fit["countrycode"] == cc].iloc[0]
    rank = int(hi_fit["resid"].rank().loc[hi_fit["countrycode"] == cc].iloc[0])
    print(
        f"  {name}: relp={row['relp']:.2f}  reli={row['reli']:.2f}  "
        f"resid={row['resid']:+.3f} ({row['resid_pct']:+.1f}%)  順位 {rank}/{len(hi_fit)}（残差が低いほど所得水準対比で物価水準が低い）"
    )

高所得クラブ（2023年）: slope=1.013（HC3 SE=0.151）  R2=0.622  n=40
  日本: relp=0.72  reli=0.58  resid=+0.165 (+17.9%)  順位 32/40（残差が低いほど所得水準対比で物価水準が低い）
  韓国: relp=0.66  reli=0.70  resid=-0.119 (-11.2%)  順位 8/40（残差が低いほど所得水準対比で物価水準が低い）
  台湾: relp=0.44  reli=0.87  resid=-0.732 (-51.9%)  順位 1/40（残差が低いほど所得水準対比で物価水準が低い）


## 2b. 感度分析：current PPP（cgdpe）による2023年横断面（2026-09-23追加）

PWT公式ドキュメントは、単年の国際的な生活水準比較にはcurrent PPPの`CGDP^e`（本稿の
`cgdpe_pc`）、国・時点をまたぐ比較にはchained PPPの`RGDP^e`（本稿の`rgdpe_pc`）を
用途として案内している。§2の主分析は2005〜2023年のパネル回帰との一貫性を優先して
`rgdpe_pc`（`reli`）を単年でも用いているが、単年比較としてはPWT公式が推奨する
`cgdpe_pc`（`reli_c`）でも結果が変わらないかを確認する。

In [4]:
hi_fit_cgdpe = fit_penn_regression(cross_2023_hi, x_col="reli_c", y_col="relp")
hi_se_hc3_cgdpe = hc3_se(hi_fit_cgdpe["log_x"].to_numpy(), hi_fit_cgdpe["log_y"].to_numpy())
print(f"高所得クラブ（2023年、current PPP）: slope={hi_fit_cgdpe.attrs['slope']:.3f}（HC3 SE={hi_se_hc3_cgdpe:.3f}）  n={len(hi_fit_cgdpe)}")

for cc, name in FOCUS_COUNTRIES.items():
    row_r = hi_fit.loc[hi_fit["countrycode"] == cc].iloc[0]
    row_c = hi_fit_cgdpe.loc[hi_fit_cgdpe["countrycode"] == cc].iloc[0]
    rank_r = int(hi_fit["resid"].rank().loc[hi_fit["countrycode"] == cc].iloc[0])
    rank_c = int(hi_fit_cgdpe["resid"].rank().loc[hi_fit_cgdpe["countrycode"] == cc].iloc[0])
    print(f"  {name}: chained PPP {row_r['resid_pct']:+.1f}%（{rank_r}位） / "
          f"current PPP {row_c['resid_pct']:+.1f}%（{rank_c}位）")

cgdpe_sensitivity = pd.DataFrame([
    {"income_measure": "rgdpe_pc（chained PPP、主分析）", "slope": hi_fit.attrs["slope"], "se_hc3": hi_se_hc3, "n": len(hi_fit),
     **{f"{cc}_resid_pct": hi_fit.loc[hi_fit["countrycode"] == cc, "resid_pct"].iloc[0] for cc in FOCUS_COUNTRIES}},
    {"income_measure": "cgdpe_pc（current PPP、感度分析）", "slope": hi_fit_cgdpe.attrs["slope"], "se_hc3": hi_se_hc3_cgdpe, "n": len(hi_fit_cgdpe),
     **{f"{cc}_resid_pct": hi_fit_cgdpe.loc[hi_fit_cgdpe["countrycode"] == cc, "resid_pct"].iloc[0] for cc in FOCUS_COUNTRIES}},
])
cgdpe_sensitivity.to_csv(OUTPUT_TABLES_DIR / "penn_regression_2023_cgdpe_sensitivity.csv", index=False)

高所得クラブ（2023年、current PPP）: slope=1.012（HC3 SE=0.151）  n=40
  日本: chained PPP +17.9%（32位） / current PPP +18.0%（32位）
  韓国: chained PPP -11.2%（8位） / current PPP -11.2%（8位）
  台湾: chained PPP -51.9%（1位） / current PPP -51.9%（1位）


## 3. プール回帰（2005〜2023年、高所得クラブ）：標準誤差の比較

同じ国を最大19回観測しているため、通常のHC3では標準誤差を過小評価する可能性がある。
国クラスタ、国×年の二方向クラスタでも再計算する。

In [5]:
pool_fit = fit_penn_regression(pool)
se_classical = classical_se(pool_fit["log_x"].to_numpy(), pool_fit["log_y"].to_numpy())
se_hc3 = hc3_se(pool_fit["log_x"].to_numpy(), pool_fit["log_y"].to_numpy())
se_cluster = cluster_se(pool_fit, "countrycode")
se_twoway = twoway_cluster_se(pool_fit, "countrycode", "year")

print(f"プール回帰: slope={pool_fit.attrs['slope']:.3f}  R2={pool_fit.attrs['r_squared']:.3f}  n={len(pool_fit)}")
print(f"傾きの標準誤差:")
print(f"  古典的        = {se_classical:.4f}")
print(f"  HC3           = {se_hc3:.4f}")
print(f"  国クラスタ    = {se_cluster:.4f}")
print(f"  国×年二方向  = {se_twoway:.4f}")

se_summary = pd.DataFrame([
    {"se_type": "classical", "se": se_classical},
    {"se_type": "HC3", "se": se_hc3},
    {"se_type": "country_cluster", "se": se_cluster},
    {"se_type": "twoway_cluster", "se": se_twoway},
])
se_summary.to_csv(OUTPUT_TABLES_DIR / "pool_regression_se_comparison.csv", index=False)

プール回帰: slope=0.988  R2=0.454  n=621
傾きの標準誤差:
  古典的        = 0.0435
  HC3           = 0.0424
  国クラスタ    = 0.1198
  国×年二方向  = 0.1161


## 4. 年固定効果付きプール回帰

年ごとに共通する変動（世界的な物価・所得ショック）を年ダミーで吸収した上で、
所得と物価水準の関係を推定する。

In [6]:
pool_fe_fit, pool_fe_result = fit_penn_regression_with_year_fe(pool)
slope_fe = pool_fe_result.params["log_x"]
se_fe = pool_fe_result.bse["log_x"]
print(f"年固定効果付きプール回帰: slope={slope_fe:.3f}（国クラスタSE={se_fe:.4f}）  n={len(pool_fe_fit)}")

print()
print("日本・韓国・台湾の年別残差（年固定効果付き）:")
for cc, name in FOCUS_COUNTRIES.items():
    sub = pool_fe_fit[pool_fe_fit["countrycode"] == cc].sort_values("year")
    first, last = sub.iloc[0], sub.iloc[-1]
    print(f"  {name}: {int(first['year'])}年 {first['resid_pct']:+.1f}% → {int(last['year'])}年 {last['resid_pct']:+.1f}%")

pool_fe_fit.to_csv(PROCESSED_DIR / "penn_regression_pooled_year_fe.csv", index=False)

年固定効果付きプール回帰: slope=0.915（国クラスタSE=0.1108）  n=621

日本・韓国・台湾の年別残差（年固定効果付き）:
  日本: 2005年 +11.4% → 2023年 +16.3%
  韓国: 2005年 -12.6% → 2023年 -10.8%
  台湾: 2005年 -54.0% → 2023年 -50.6%


## 5. 年別の横断面回帰（プール回帰に依存しない時系列の確認）

年ごとに独立した横断面回帰を行い、各年の日本・台湾の残差を直接比較する。
「収束」「乖離拡大」という時系列の主張は、この年別回帰で検証する。

In [7]:
# 高所得クラブと同じ対象年（2005-2023）・国基準に絞って年別回帰する
EXCL = {
    "IRL", "LUX", "QAT", "ARE", "KWT", "BRN", "SGP", "MAC", "MLT", "CYM", "BMU", "VGB",
    "ABW", "SXM", "AIA", "MSR", "KNA", "ATG", "SYC", "TCA", "GUY", "TTO", "BHS", "SAU",
    "OMN", "BHR", "KAZ", "TKM", "BLR", "RUS",
}
pwt_all = pd.read_csv(PROCESSED_DIR / "pwt_relative_price_income.csv")
pwt_hi_panel = pwt_all[
    pwt_all["year"].between(2005, 2023)
    & (pwt_all["rgdpe_pc"] >= 30000)
    & (~pwt_all["countrycode"].isin(EXCL))
    & (pwt_all["relp"] > 0) & (pwt_all["reli"] > 0)
]
by_year_fit = fit_penn_regression_by_year(pwt_hi_panel)

year_summary_rows = []
for year, group in by_year_fit.groupby("fit_year"):
    row = {"year": int(year), "slope": group["fit_slope"].iloc[0], "n": len(group)}
    for cc in FOCUS_COUNTRIES:
        match = group[group["countrycode"] == cc]
        row[f"{cc}_resid_pct"] = match["resid_pct"].iloc[0] if len(match) else np.nan
    year_summary_rows.append(row)
year_summary_df = pd.DataFrame(year_summary_rows).sort_values("year")
print(year_summary_df.round(2).to_string(index=False))

by_year_fit.to_csv(PROCESSED_DIR / "penn_regression_by_year.csv", index=False)
year_summary_df.to_csv(OUTPUT_TABLES_DIR / "penn_regression_by_year_summary.csv", index=False)

 year  slope  n  JPN_resid_pct  KOR_resid_pct  TWN_resid_pct
 2005   0.62 26          11.55         -17.29         -54.03
 2006   0.73 28           5.69          -8.19         -54.10
 2007   0.67 28          -3.36         -12.50         -57.55
 2008   0.78 30           8.07         -23.78         -54.72
 2009   0.66 28          24.54         -31.07         -55.74
 2010   0.72 29          28.08         -24.61         -56.14
 2011   0.77 31          37.45         -19.64         -54.39
 2012   0.91 31          45.64         -12.44         -53.98
 2013   0.85 31          10.61          -4.38         -54.83
 2014   0.84 31           7.28          -2.23         -55.79
 2015   1.03 32          10.59           1.86         -51.91
 2016   1.12 33          34.47          -0.47         -50.10
 2017   1.01 36          29.83           1.65         -46.23
 2018   1.00 37          30.23          -2.13         -45.92
 2019   1.02 38          48.93           1.59         -44.02
 2020   0.97 35         

## 5b. 感度分析：2023年時点の40か国構成を固定した年別回帰（2026-09-23追加）

§5の年別回帰は、各年についてその年の一人当たりGDPが3万ドル以上かどうかを判定し
直すため、対象国数が2005年26か国→2023年40か国と年によって変わる。国の入れ替わり
自体が残差の年次変化に混入しないよう、**2023年時点で高所得クラブに含まれる40か国に
対象国を固定**した上で、2005〜2023年の全期間について年別回帰を行う。

In [8]:
countries_2023 = set(cross_2023_hi["countrycode"])
pwt_fixed_panel = pwt_all[
    pwt_all["year"].between(2005, 2023)
    & pwt_all["countrycode"].isin(countries_2023)
    & (pwt_all["relp"] > 0) & (pwt_all["reli"] > 0)
]
by_year_fixed_fit = fit_penn_regression_by_year(pwt_fixed_panel)

year_fixed_rows = []
for year, group in by_year_fixed_fit.groupby("fit_year"):
    row = {"year": int(year), "slope": group["fit_slope"].iloc[0], "n": len(group)}
    for cc in FOCUS_COUNTRIES:
        match = group[group["countrycode"] == cc]
        row[f"{cc}_resid_pct"] = match["resid_pct"].iloc[0] if len(match) else np.nan
    year_fixed_rows.append(row)
year_fixed_df = pd.DataFrame(year_fixed_rows).sort_values("year")
print(f"固定標本（2023年の{len(countries_2023)}か国）での年別回帰:")
print(year_fixed_df.round(2).to_string(index=False))

print()
print("各年の判定標本（現稿・主分析）との比較（2005年・2023年）:")
for year in (2005, 2023):
    dyn = year_summary_df[year_summary_df["year"] == year].iloc[0]
    fix = year_fixed_df[year_fixed_df["year"] == year].iloc[0]
    print(f"  {year}年: 各年判定 n={int(dyn['n'])} JPN={dyn['JPN_resid_pct']:+.1f}% TWN={dyn['TWN_resid_pct']:+.1f}%  |  "
          f"固定標本 n={int(fix['n'])} JPN={fix['JPN_resid_pct']:+.1f}% TWN={fix['TWN_resid_pct']:+.1f}%")

by_year_fixed_fit.to_csv(PROCESSED_DIR / "penn_regression_by_year_fixed_membership.csv", index=False)
year_fixed_df.to_csv(OUTPUT_TABLES_DIR / "penn_regression_by_year_fixed_membership_summary.csv", index=False)

固定標本（2023年の40か国）での年別回帰:


 year  slope  n  JPN_resid_pct  KOR_resid_pct  TWN_resid_pct
 2005   0.78 40          13.39         -13.33         -53.24
 2006   0.79 40           6.54          -6.63         -53.76
 2007   0.75 40          -2.28         -10.66         -57.21
 2008   0.74 40           7.21         -24.73         -55.06
 2009   0.79 40          26.87         -29.18         -55.28
 2010   0.84 40          29.86         -23.16         -56.05
 2011   0.89 40          39.59         -17.90         -54.36
 2012   0.92 40          44.98         -12.77         -54.27
 2013   0.90 40          10.79          -3.83         -55.04
 2014   0.92 40           8.31          -0.92         -56.04
 2015   1.02 40          10.35           1.62         -51.95
 2016   1.08 40          33.35          -1.12         -50.11
 2017   1.09 40          31.99           2.76         -46.24
 2018   1.07 40          32.03          -1.49         -45.92
 2019   1.08 40          50.66           2.10         -44.08
 2020   1.02 40        

## 6. トリム回帰（外れ値への頑健性）

残差が標準偏差の2.5倍を超える国を反復的に除外して係数を再推定する。

In [9]:
x = np.log(cross_2023_hi["reli"].to_numpy())
y = np.log(cross_2023_hi["relp"].to_numpy())
b_trim, resid_trim, keep = trim_ols(x, y)

excluded = cross_2023_hi.loc[~keep, "country"].tolist()
print(f"トリム回帰の傾き: {b_trim[1]:.3f}（通常回帰: {hi_fit.attrs['slope']:.3f}）")
print(f"除外された国（{len(excluded)}か国）: {excluded}")

トリム回帰の傾き: 1.088（通常回帰: 1.013）
除外された国（1か国）: ['Taiwan']


## 7. 所得閾値を変えた頑健性チェック

高所得クラブの定義を2.5万〜3.5万ドルの範囲で動かす。

In [10]:
pwt_2023 = cross_2023_world.copy()
threshold_rows = []
for threshold in (25000, 30000, 35000):
    subset = pwt_2023[
        (pwt_2023["rgdpe_pc"] >= threshold) & (~pwt_2023["countrycode"].isin(EXCL))
    ].copy()
    fit = fit_penn_regression(subset)
    row = {"threshold": threshold, "n": len(fit), "slope": fit.attrs["slope"]}
    for cc in FOCUS_COUNTRIES:
        row[f"{cc}_resid_pct"] = fit.loc[fit["countrycode"] == cc, "resid_pct"].iloc[0]
    threshold_rows.append(row)

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.round(2).to_string(index=False))

 threshold  n  slope  JPN_resid_pct  KOR_resid_pct  TWN_resid_pct
     25000 46   0.79          11.57         -12.42         -50.20
     30000 40   1.01          17.94         -11.23         -51.89
     35000 37   0.85          12.13         -12.93         -51.10


## 8. 除外国リストへの感度分析

事前に定めた除外国リスト（タックスヘイブン・産油国・非市場経済等30か国）を
適用しない場合（所得閾値のみで選定）と比較する。除外リストの妥当性を
検証したものではなく、**結果がこのリストにどの程度依存しているか**を示す。

In [11]:
no_excl = pwt_2023[pwt_2023["rgdpe_pc"] >= 30000].copy()
fit_no_excl = fit_penn_regression(no_excl)

print(f"除外リストあり（現行）: slope={hi_fit.attrs['slope']:.3f}  n={len(hi_fit)}")
print(f"除外リストなし（3万ドル以上の全{len(fit_no_excl)}か国）: slope={fit_no_excl.attrs['slope']:.3f}  n={len(fit_no_excl)}")
print()
for cc, name in FOCUS_COUNTRIES.items():
    with_excl = hi_fit.loc[hi_fit["countrycode"] == cc, "resid_pct"].iloc[0]
    without_excl = fit_no_excl.loc[fit_no_excl["countrycode"] == cc, "resid_pct"].iloc[0]
    rank_with = int(hi_fit["resid"].rank().loc[hi_fit["countrycode"] == cc].iloc[0])
    rank_without = int(fit_no_excl["resid"].rank().loc[fit_no_excl["countrycode"] == cc].iloc[0])
    print(f"  {name}: 除外リストあり {with_excl:+.1f}%（{rank_with}/{len(hi_fit)}位） / "
          f"なし {without_excl:+.1f}%（{rank_without}/{len(fit_no_excl)}位）（差 {without_excl - with_excl:+.1f}pt）")

no_excl_ranked = fit_no_excl.sort_values("resid")
print()
print("除外リストなし64か国での残差下位5か国（残差が小さい順）:")
print(no_excl_ranked[["country", "countrycode", "resid_pct"]].head(5).to_string(index=False))

excl_sensitivity = pd.DataFrame([
    {"spec": "with_excl_list", "slope": hi_fit.attrs["slope"], "n": len(hi_fit),
     **{f"{cc}_resid_pct": hi_fit.loc[hi_fit["countrycode"] == cc, "resid_pct"].iloc[0] for cc in FOCUS_COUNTRIES},
     **{f"{cc}_rank": int(hi_fit["resid"].rank().loc[hi_fit["countrycode"] == cc].iloc[0]) for cc in FOCUS_COUNTRIES}},
    {"spec": "no_excl_list", "slope": fit_no_excl.attrs["slope"], "n": len(fit_no_excl),
     **{f"{cc}_resid_pct": fit_no_excl.loc[fit_no_excl["countrycode"] == cc, "resid_pct"].iloc[0] for cc in FOCUS_COUNTRIES},
     **{f"{cc}_rank": int(fit_no_excl["resid"].rank().loc[fit_no_excl["countrycode"] == cc].iloc[0]) for cc in FOCUS_COUNTRIES}},
])
excl_sensitivity.to_csv(OUTPUT_TABLES_DIR / "exclusion_list_sensitivity.csv", index=False)

除外リストあり（現行）: slope=1.013  n=40
除外リストなし（3万ドル以上の全64か国）: slope=0.552  n=64

  日本: 除外リストあり +17.9%（32/40位） / なし +17.8%（46/64位）（差 -0.1pt）
  韓国: 除外リストあり -11.2%（8/40位） / なし -3.2%（25/64位）（差 +8.0pt）
  台湾: 除外リストあり -51.9%（1/40位） / なし -42.1%（3/64位）（差 +9.8pt）

除外リストなし64か国での残差下位5か国（残差が小さい順）:
           country countrycode  resid_pct
 Brunei Darussalam         BRN -62.506504
Russian Federation         RUS -48.458577
            Taiwan         TWN -42.070956
        Kazakhstan         KAZ -41.881958
           Türkiye         TUR -41.671915


## 9. 保存

In [12]:
hi_fit.to_csv(PROCESSED_DIR / "penn_regression_2023_highincome.csv", index=False)
pool_fit.to_csv(PROCESSED_DIR / "penn_regression_pooled_2005_2023.csv", index=False)
threshold_df.to_csv(OUTPUT_TABLES_DIR / "penn_regression_threshold_robustness.csv", index=False)

summary = pd.DataFrame([
    {"spec": "世界全体2023", "slope": world_fit.attrs["slope"], "se": np.nan, "n": len(world_fit)},
    {"spec": "高所得クラブ2023(HC3)", "slope": hi_fit.attrs["slope"], "se": hi_se_hc3, "n": len(hi_fit)},
    {"spec": "プール2005-2023(国クラスタ)", "slope": pool_fit.attrs["slope"], "se": se_cluster, "n": len(pool_fit)},
    {"spec": "プール2005-2023(二方向クラスタ)", "slope": pool_fit.attrs["slope"], "se": se_twoway, "n": len(pool_fit)},
    {"spec": "プール+年固定効果", "slope": slope_fe, "se": se_fe, "n": len(pool_fe_fit)},
    {"spec": "トリム2023", "slope": b_trim[1], "se": np.nan, "n": int(keep.sum())},
    {"spec": "除外リストなし2023", "slope": fit_no_excl.attrs["slope"], "se": np.nan, "n": len(fit_no_excl)},
])
summary.to_csv(OUTPUT_TABLES_DIR / "penn_regression_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

                 spec  slope    se   n
             世界全体2023  0.135   NaN 185
      高所得クラブ2023(HC3)  1.013 0.151  40
  プール2005-2023(国クラスタ)  0.988 0.120 621
プール2005-2023(二方向クラスタ)  0.988 0.116 621
            プール+年固定効果  0.915 0.111 621
              トリム2023  1.088   NaN  39
          除外リストなし2023  0.552   NaN  64


## まとめと限界

- **標準誤差**：同一国を最大19回観測するプール回帰では、国クラスタ・二方向クラスタの
  標準誤差がHC3より大きい。国クラスタを基本とする。
- **所得指標の頑健性（2026-09-23追加、§2b）**：2023年単年の横断面は、PWT公式が
  年またぎ比較用に案内する`rgdpe_pc`（chained PPP）を主分析としているが、PWT公式が
  単年比較用に案内する`cgdpe_pc`（current PPP）でもほぼ同じ結果になる（傾き・
  残差・順位のいずれも大きな差はない）。
- **時系列の主張**：「日本は収束、台湾は乖離拡大」という主張を、年固定効果付き
  プール回帰（§4）と年別横断面回帰（§5）の両方で検証した。年効果なしプール
  回帰の2時点比較だけでは世界共通の変動と各国固有の変動を区別できず、実際に
  両仕様で確認した結果、両国とも年次変動が大きく、単調な収束・乖離拡大の
  トレンドは確認されなかった。この主張は撤回する。
- **標本構成の固定（2026-09-23追加、§5b）**：§5の年別回帰は毎年の所得閾値判定に
  よって対象国数が変わる（2005年26か国→2023年40か国）。2023年時点の40か国に
  対象を固定して年別回帰をやり直しても、台湾が一貫して大きく負である傾向は
  変わらない。ただし日本の残差の水準は標本の定義（毎年判定か、2023年基準に
  固定か）に一定の感度を持つ。
- **除外国リストへの依存**：除外リストの有無で傾き・残差の大きさが変わる（§8）。
  **除外リストありの高所得クラブ40か国では台湾の残差が最小（1位）だが、除外リスト
  なしの64か国では3位**（ブルネイ・ロシアに次ぐ）になる。本文・結論では
  「除外リスト適用後の40か国中最小」であることを常に明示し、「もっとも割安な国」
  のように限定なしで述べない。
- **理論的解釈**：Balassa-Samuelson理論が予測するのは所得と物価水準の正の関係で
  あり、傾きが1になることではない。本稿は「傾きが正で得られた」という以上の
  理論的含意を主張しない。
- **識別の限界**：所得と物価水準の相関の記述であり、因果関係を識別するものではない。
- **台湾のデータについて**：台湾はWorld Bankの通常の国リストにない。Penn World
  Tableにおける台湾のデータソース・検証プロセスの詳細は、公式ドキュメントで
  個別に確認する必要がある（本稿では未確認）。